# DS 227 · Week 11 — Ethics and privacy
## Re-identification, k-anonymity and differential privacy

Every code block from the lecture, in order, so you can run what you watched instead of
retyping it. The first slides use small tables typed into the cells; the **Case study**
cell loads the flood control data from the course API, so there is no file to download.

**How to use this.** Run the cells top to bottom the first time (Runtime → Run all). The
exercise cells list their tasks above the code: predict first, then run, then change one
thing and run again.

**Version differences.** The slides follow pandas 3. On pandas 2, text columns show as
`object` instead of `str`, and dates as `datetime64[ns]` instead of `datetime64[us]`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## The lecture, cell by cell

Headings carry the slide numbers shown in the deck's counter.


### Slide 7 — Code on these slides runs in the browser


Expected output:

```
pandas version: 3.0.5
64-bit Python: True
city      str
pop     int64
dtype: object
```


In [ ]:
import io, sys

print("pandas version:", pd.__version__)
print("64-bit Python:", sys.maxsize > 2**32)

t = pd.read_csv(io.StringIO("city,pop\nCebu,964000"))
print(t.dtypes)

### Slide 20 — Direct, quasi and sensitive columns


Expected output:

```
student_id  age barangay   gpa
0   2024-001   19    Lahug  1.75
1   2024-002   19    Lahug  2.00
2   2024-003   21     Apas  1.25
3   2024-004   19    Lahug  1.50
```


In [ ]:
# the lab's four students (the lab calls this table df)
students = pd.DataFrame({
    "student_id": ["2024-001", "2024-002", "2024-003", "2024-004"],
    "age": [19, 19, 21, 19],
    "barangay": ["Lahug", "Lahug", "Apas", "Lahug"],
    "gpa": [1.75, 2.00, 1.25, 1.50],
})
print(students)

### Slide 22 — k-anonymity: counting group sizes


**Exercise.**

1. Predict the size of each age-and-barangay group.
2. Run the cell.
3. Group by ["barangay"] only. Is anyone still alone?


In [ ]:
# students: the lab's four students
sizes = students.groupby(["age", "barangay"]).size()
print(sizes)

### Slide 24 — The flood data, audited for re-identification


Expected output:

```
(34079, 15)
```


In [ ]:
import io, requests

url = "https://portal.latarak.com/api/flood/export.csv"
df  = pd.read_csv(io.StringIO(requests.get(url, timeout=60).text))
print(df.shape)

### Slide 25 — Region alone identifies no one


Expected output:

```
18
180
0
```


In [ ]:
g = df.groupby(["region"]).size()

print(len(g))          # groups
print(g.min())         # the smallest group: k
print((g == 1).sum())  # rows alone in their group

### Slide 27 — Three lines, wrapped in a function


Expected output:

```
16,646 groups, 10,534 alone (30.9%)
```


In [ ]:
def audit(cols):
    g = df.groupby(cols).size()
    alone = (g == 1).sum()
    print(f"{len(g):,} groups, {alone:,} alone ({alone / len(df):.1%})")
qi = ["region", "province", "year", "b"]

# precise: PHP millions
df["b"] = (df.budget / 1e6).round()
audit(qi)

### Slide 28 — Generalisation restores privacy


Expected output:

```
4,675 groups, 915 alone (2.7%)
821 groups, 132 alone (0.4%)
```


In [ ]:
# audit(cols), qi: from the previous slide
# generalised: PHP 50M bands
df["b"] = (df.budget / 5e7).round() * 50
audit(qi)

# and generalise place: region, not province
audit(["region", "year", "b"])

### Slide 30 — Dropping direct identifiers and generalizing quasi-identifiers


Expected output:

```
barangay   gpa age_band
0    Lahug  1.75    18-20
1    Lahug  2.00    18-20
2     Apas  1.25    21-25
3    Lahug  1.50    18-20
```


In [ ]:
# students: the lab's four students
safe = students.drop(columns=["student_id"])
safe["age_band"] = pd.cut(safe["age"], bins=[17, 20, 25],
                          labels=["18-20", "21-25"])
safe = safe.drop(columns=["age"])
print(safe)

### Slide 32 — A mean over one record reveals that record


**Exercise.**

1. Predict Apas’s mean GPA, and what it reveals.
2. Run the cell.
3. Add .query("count >= 2") to the end. What is published now?


In [ ]:
# students: the lab's four students
print(students.groupby("barangay")["gpa"].agg(["mean", "count"]))

### Slide 33 — The smallest real cell in this dataset


Expected output:

```
region                            year
Cordillera Administrative Region  2026     1
Central Office                    2024     6
2020    13
dtype: int64
180 179
```


In [ ]:
c = df.groupby(["region", "year"]).size()
print(c.nsmallest(3))

# suppress cells below a threshold
safe = c[c >= 5]
print(len(c), len(safe))

### Slide 35 — A solution to the repeated-query problem


**Exercise.**

1. Predict which privacy budget (eps) gives the noisiest answers.
2. Run the cell.
3. Change size=3 to size=1000 and print noisy.mean().round(1). What happens to the noise?


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
true_count = 1263                       # revoked-firm projects (week 9)

for eps in [0.1, 1.0, 10.0]:            # the privacy budget per query
    noisy = true_count + rng.laplace(scale=1 / eps, size=3)
    print(eps, noisy.round(1))

### Slide 37 — Hashing hides the name, not the person


Expected output:

```
3e0f221a8d0c
```


In [ ]:
import hashlib

print(hashlib.sha256(b"2024-001").hexdigest()[:12])

---

*Data: DPWH flood control projects 2016–2026, mirrored by BetterGov.ph from the DPWH
transparency portal, CC0-1.0 (public domain). Served by the course API at* `/api/flood`.
